# 08 · Einen GaSi-Forschungsablauf hochladen

**Aufgabe:** Einen identifizierten Gas-/Gasfeuchtesensor mit Messplan, Messung und Auswertung nachvollziehbar ablegen.

**Run All bleibt lokal:** Die fehlende Laufvorlage wird einmal angelegt und die Uploadbereitschaft geprüft. Originaldateien bleiben unverändert. Es wird nichts übertragen.

1. `config/gasi_run.json` mit Sensor-ID, Ressourcen, Stufen, Messorten, Kanalzuordnung, Daten- und Stufenbereichen (source_ranges/stage_ranges), Zeiten mit UTC-Offset sowie ausgewählten Dateien vervollständigen.
2. In der zentralen `config/playground_test_target.json` den Abschnitt `gasi_workflow` ausdrücklich konfigurieren. Die Ziele werden nicht aus dem synthetischen Test übernommen.
3. Notebook 04 prüft Typen und Wertelisten lesend. Neue Sensoren zuerst mit geprüftem `TEST_OBJECT`-/`SAMPLE`-Typ registrieren oder ihre vorhandene `INSTRUMENT`-Identität verwenden.
4. Erst nach vollständiger Prüfung die letzte Zelle bewusst aktivieren. Der Upload liest Host, Collections, Ressourcen und Pflichtfelder erneut, ergänzt Teilstände und verifiziert alle Dateien per Rückdownload/SHA-256.

Die Laufkennung bleibt bei Wiederholung gleich. Eine neue Auswertung bekommt eine neue `analysis_id` und verwendet den bestehenden Messlauf. Nicht gleichzeitig aus mehreren Kernels schreiben.

Weitere Details: [GaSi-Forschungsablauf](docs/GaSi_Forschungsupload.md).

In [ ]:
from pathlib import Path
import importlib
import gasi_run
from IPython.display import FileLink, display

importlib.invalidate_caches()
gasi_run = importlib.reload(gasi_run)
run_path = Path("config/gasi_run.json")
if not run_path.exists():
    with run_path.open("xb") as target:
        target.write(Path("config/gasi_run.example.json").read_bytes())
    print("Leere lokale Laufvorlage angelegt: config/gasi_run.json")
preview_sha256 = None
run = gasi_run.load_run(run_path)
issues = gasi_run.validate_run(run)
target_config, target_message = gasi_run.ob.load_target_config()
print(target_message)
for activity in gasi_run.ACTIVITIES:
    try:
        path = gasi_run.ob.target_path((target_config or {}).get("gasi_workflow", {}).get("targets", {}).get(activity, {}))
        print(activity + ":", path)
    except gasi_run.ob.CheckFailed:
        print(activity + ": NICHT KONFIGURIERT")
try:
    gasi_run._targets(target_config)
except gasi_run.ob.CheckFailed as exc:
    issues.append("Lokale Zielprüfung: " + str(exc))
if issues:
    print(f"Upload gesperrt: {len(issues)} Angaben fehlen oder sind zu prüfen.")
    for issue in issues:
        print("-", issue)
else:
    bundle = gasi_run.prepare_bundle(run)
    preview_sha256 = bundle["sha256"]
    print("Lokales Uploadpaket geprüft; noch keine Instanzbestätigung.")
    print("Tätigkeiten: Vorbereitung, Beaufschlagung/Messung, Auswertung.")
    print("Prüfling und Ressourcen werden als vorhandene Objekte wiederverwendet.")
    print("Alle Originaldateien bleiben unverändert.")
    display(FileLink((bundle["path"] / "manifest.json").relative_to(Path.cwd()).as_posix(), result_html_prefix="Lokales Manifest: "))

In [ ]:
# Bewusst aktivieren und nur diese Zelle ausführen, wenn Lauf und Ziele geprüft sind.
UPLOAD_ENABLED = False
CONFIRMATION = ""  # UPLOAD CONFIGURED GASI RESEARCH WORKFLOW TO PLAYGROUND

result, message = gasi_run.upload_workflow(enabled=UPLOAD_ENABLED, confirmation=CONFIRMATION, expected_manifest_sha=globals().get("preview_sha256"))
print(message)
if result:
    for activity, entry in result["objects"].items():
        print(activity, entry["identifier"], entry["collection"], entry.get("url", ""))
    for entry in result["datasets"]:
        print(entry["activity"], entry["identifier"], "SHA-256:", entry["sha256"], entry.get("url", ""))